### Explanation of Embeddings


In [ ]:
import numpy as np 
# NumPy is Python’s most popular library for numerical computing.
#It provides a powerful object called an ndarray (N-dimensional array)
#We use numpy to store embeddings as number as numpy are faster 

from sentence_transformers import SentenceTransformer
#It is used to convert text into embeddings using hugging face model - "all-MiniLM-L6-v2"

import chromadb
from chromadb.config import Settings
#Use to configure chroma db the vectorDb 

import uuid
#Universally unique identifier - every emebedding is given a unique identity 

from typing import List, Dict, Any, Tuple
#These don’t affect execution.
#They help humans and editors understand what a function expects and returns.

from sklearn.metrics.pairwise import cosine_similarity
#Use to find similarity between diffrent data like i ask a Q - "why is burj khalifa so tall" && "Why was burj khalifa made so tall" so these are similar that will be found by cosine similarity 

In [ ]:
class EmbeddingHead:
    """Handles document embedding generation using SentenceTransformer"""  # Class responsible for loading an embedding model and generating text embeddings.

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding head.

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """

        self.model_name = model_name      # Store the embedding model name inside the object.
        self.model = None                 # Initially, no model is loaded into memory.
        self._load_model()                # Automatically load the model when an object is created.

    def _load_model(self):
        """Load the SentenceTransformer model."""

        try:
            print(f"Loading embedding model: {self.model_name}")  # Display which model is being loaded.

            self.model = SentenceTransformer(self.model_name)     # Download/load the embedding model from Hugging Face.

            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}"
            )                                                    # Print the embedding vector size (e.g., 384).

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")  # Display the actual error message.
            raise                                                 # Re-raise the exception to stop execution.

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:    #Convert text into embeddings return type is numpy array
        
        """Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape
            (number_of_texts, embedding_dimension).
        """

        if not self.model:
            raise ValueError("Model not loaded")                  # Prevent embedding generation if the model failed to load.

        print(f"Generating embeddings for {len(texts)} texts...") # Display the number of texts being processed.

        embeddings = self.model.encode(                           # Encode is the fnc used on texts 
            texts,
            show_progress_bar=True
        )                                                         # Convert each text into a numerical embedding vector.

        print(f"Generated embeddings with shape: {embeddings.shape}")  # Example: (25, 384)

        return embeddings                                         # Return the generated embedding vectors.


# -------------------- Example Usage --------------------

embeddings = EmbeddingHead()      # Create an EmbeddingHead object. This automatically loads the embedding model.

embeddings                        # Display the created object in Jupyter Notebook.

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""
    # This class handles all vector database operations such as creating, storing, and managing document embeddings using ChromaDB.

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the Vector Store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory where the vector database will be stored. i.e. stored on the hard disk
        """

        self.collection_name = collection_name      # Store the collection name so it can be used throughout the class.

        self.persist_directory = persist_directory  # Save the directory path where ChromaDB will persist all embeddings.

        self.client = None                          # Placeholder for the ChromaDB client. It will be initialized later.

        self.collection = None                      # Placeholder for the collection that will hold document embeddings.

        self._initialize_store()                    # Automatically create/connect to the vector store during object creation.